# Building an xBA model

The first thing I will do is create a dataset where each row represents a batted ball, and the target is whether or not the outcome was a hit. Batted balls include: single, double, triple, home run, field out, force out, double play, error, and fielders choice. The model will learn the relationship between contact quality and outcomes.

The initial development sample covers the 2024 regular season from April 1 through September 29. Training uses batted balls through September 22, while September 23 through September 29 is held out as a chronological test window. Features include launch speed (exit velocity), launch angle, spray direction, batter/pitcher handedness, and pitch information.

In [13]:
# Step 1: Download a full-season Statcast sample for development, then cache it locally.
# The cache prevents repeated API requests every time the notebook is re-run.
from pathlib import Path

import numpy as np
import pandas as pd
from pybaseball import statcast

START_DATE = "2024-04-01"
END_DATE = "2024-09-29"
TEST_START_DATE = "2024-09-23"
CACHE_DIR = Path("data")
CACHE_PATH = CACHE_DIR / f"statcast_{START_DATE}_{END_DATE}.parquet"

if CACHE_PATH.exists():
    # Reuse the existing extract when this exact date range has already been downloaded.
    raw = pd.read_parquet(CACHE_PATH)
    print(f"Loaded {len(raw):,} pitches from {CACHE_PATH}")
else:
    # Download the date range once and save the raw response for reproducibility.
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    raw = statcast(START_DATE, END_DATE)
    raw.to_parquet(CACHE_PATH, index=False)
    print(f"Downloaded and cached {len(raw):,} pitches to {CACHE_PATH}")

# Step 2: Keep only events where the ball was put into play and define the hit target.
batted_ball_events = {
    "single", "double", "triple", "home_run", "field_out",
    "force_out", "grounded_into_double_play", "field_error",
    "fielders_choice", "fielders_choice_out", "double_play"
}

contact = raw.loc[
    raw["events"].isin(batted_ball_events),
    [
        "game_date", "batter", "events", "launch_speed", "launch_angle",
        "hc_x", "hc_y", "stand", "p_throws", "pitch_type",
    ],
].copy()

# Statcast does not provide a direct launch-direction column in this extract.
# Convert hit coordinates into an approximate spray angle relative to center field.
contact["spray_angle_degrees"] = np.degrees(
    np.arctan2(contact["hc_x"] - 125.42, 198.27 - contact["hc_y"])
)

# The target is 1 for any hit and 0 for every other retained batted-ball outcome.
contact["is_hit"] = contact["events"].isin(
    {"single", "double", "triple", "home_run"}
).astype(int)

contact.head()

Loaded 694,688 pitches from data/statcast_2024-04-01_2024-09-29.parquet


,game_date,batter,events,launch_speed,launch_angle,hc_x,hc_y,stand,p_throws,pitch_type,spray_angle_degrees,is_hit
0,2024-09-29,444482,field_out,98.1,-17,156.22,161.23,L,R,FF,39.744657,0
1,2024-09-29,630105,field_out,102.1,8,99.52,154.14,L,R,FC,-30.408772,0
4,2024-09-29,665487,field_out,100.9,48,70.57,84.97,R,R,FF,-25.832226,0
5,2024-09-29,664983,field_out,84.9,52,74.8,114.85,L,L,FF,-31.24975,0
8,2024-09-29,553993,single,102.5,15,87.83,104.64,R,R,SI,-21.874172,1


In [14]:
# Step 3: Remove rows missing features required by the first model.
# We do this after defining the target so missingness cannot silently change the label.
model_data = contact.dropna(
    subset=[
        "launch_speed", "launch_angle", "spray_angle_degrees",
        "stand", "p_throws", "pitch_type", "is_hit",
    ]
).copy()

# Add simple nonlinear terms so the model can learn sweet spots and interactions.
model_data["exit_velocity_squared"] = model_data["launch_speed"] ** 2
model_data["launch_angle_squared"] = model_data["launch_angle"] ** 2
model_data["spray_angle_squared"] = model_data["spray_angle_degrees"] ** 2
model_data["exit_velocity_x_launch_angle"] = (
    model_data["launch_speed"] * model_data["launch_angle"]
)
model_data["exit_velocity_x_spray_angle"] = (
    model_data["launch_speed"] * model_data["spray_angle_degrees"]
)
model_data["launch_angle_x_spray_angle"] = (
    model_data["launch_angle"] * model_data["spray_angle_degrees"]
)

print(f"Contact events: {len(contact):,}")
print(f"Complete modeling rows: {len(model_data):,}")
print(f"Observed batting average: {model_data['is_hit'].mean():.3f}")
print("Missing values by field:")
print(contact.isna().sum().sort_values(ascending=False).head(8))

Contact events: 119,602
Complete modeling rows: 119,193
Observed batting average: 0.325
Missing values by field:
launch_speed           363
launch_angle           301
hc_x                    48
hc_y                    48
spray_angle_degrees     48
pitch_type              45
game_date                0
batter                   0
dtype: int64


In [15]:
# Step 4: Split by date so the model is tested on the final week of the season.
# Training on all earlier dates preserves temporal order while providing a larger evaluation window.
model_data["game_date"] = pd.to_datetime(model_data["game_date"])
test_start_date = pd.Timestamp(TEST_START_DATE)
train_data = model_data[model_data["game_date"] < test_start_date].copy()
test_data = model_data[model_data["game_date"] >= test_start_date].copy()

numeric_features = [
    "launch_speed",
    "launch_angle",
    "spray_angle_degrees",
    "exit_velocity_squared",
    "launch_angle_squared",
    "spray_angle_squared",
    "exit_velocity_x_launch_angle",
    "exit_velocity_x_spray_angle",
    "launch_angle_x_spray_angle",
]
categorical_features = ["stand", "p_throws", "pitch_type"]
feature_columns = numeric_features + categorical_features

X_train = train_data[feature_columns]
y_train = train_data["is_hit"]
X_test = test_data[feature_columns]
y_test = test_data["is_hit"]

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training dates: {train_data['game_date'].min().date()} to {train_data['game_date'].max().date()}")
print(f"Test dates: {test_data['game_date'].min().date()} to {test_data['game_date'].max().date()}")
print(f"Spray angle range: {model_data['spray_angle_degrees'].min():.1f} to {model_data['spray_angle_degrees'].max():.1f} degrees")

Training rows: 114,880
Test rows: 4,313
Training dates: 2024-04-01 to 2024-09-22
Test dates: 2024-09-23 to 2024-09-29
Spray angle range: -179.9 to 179.9 degrees


In [16]:
# Step 5: Build an interpretable logistic-regression baseline.
# One-hot encoding lets the model use categorical features without treating labels as numbers.
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1_000, random_state=42)),
    ]
)

logistic_model.fit(X_train, y_train)

# Predict probabilities rather than only hit/out labels because xBA is an average of probabilities.
test_data["predicted_hit_probability"] = logistic_model.predict_proba(X_test)[:, 1]
test_data[["game_date", "batter", "is_hit", "predicted_hit_probability"]].head()

,game_date,batter,is_hit,predicted_hit_probability
0,2024-09-29,444482,0,0.231445
1,2024-09-29,630105,0,0.602386
4,2024-09-29,665487,0,0.225124
5,2024-09-29,664983,0,0.061216
8,2024-09-29,553993,1,0.579602


In [27]:
# Step 6: Build a spline-based logistic model with smooth pairwise interactions.
# Spline basis products model effects such as exit velocity x launch angle without raw quadratic assumptions.
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer

spline_numeric_features = [
    "launch_speed",
    "launch_angle",
    "spray_angle_degrees",
]

numeric_spline_interactions = Pipeline(
    steps=[
        (
            "splines",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric_splines",
                        SplineTransformer(
                            n_knots=6,
                            degree=3,
                            knots="quantile",
                            include_bias=False,
                        ),
                        spline_numeric_features,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "pairwise_interactions",
            PolynomialFeatures(
                degree=2,
                interaction_only=True,
                include_bias=False,
            ),
        ),
    ]
)

categorical_features_only = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

spline_preprocessor = FeatureUnion(
    transformer_list=[
        ("numeric_spline_interactions", numeric_spline_interactions),
        ("categorical", categorical_features_only),
    ]
)

spline_model = Pipeline(
    steps=[
        ("preprocessor", spline_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2_000,
                C=0.5,
                random_state=42,
            ),
        ),
    ]
)

spline_model.fit(X_train, y_train)
test_data["spline_predicted_hit_probability"] = spline_model.predict_proba(X_test)[:, 1]
test_data[
    ["game_date", "batter", "is_hit", "spline_predicted_hit_probability"]
].head()

,game_date,batter,is_hit,spline_predicted_hit_probability
0,2024-09-29,444482,0,0.143210
1,2024-09-29,630105,0,0.677753
4,2024-09-29,665487,0,0.098915
5,2024-09-29,664983,0,0.001131
8,2024-09-29,553993,1,0.727935


In [28]:
# Step 7: Evaluate probability quality on the held-out week.
# Log loss and Brier score reward useful probabilities; accuracy alone would hide calibration errors.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
baseline_probabilities = baseline.predict_proba(X_test)[:, 1]
model_probabilities = test_data["predicted_hit_probability"]
spline_probabilities = test_data["spline_predicted_hit_probability"]

metrics = pd.DataFrame(
    {
        "log_loss": [
            log_loss(y_test, baseline_probabilities),
            log_loss(y_test, model_probabilities),
            log_loss(y_test, spline_probabilities),
        ],
        "brier_score": [
            brier_score_loss(y_test, baseline_probabilities),
            brier_score_loss(y_test, model_probabilities),
            brier_score_loss(y_test, spline_probabilities),
        ],
        "accuracy_at_0.5": [
            accuracy_score(y_test, baseline_probabilities >= 0.5),
            accuracy_score(y_test, model_probabilities >= 0.5),
            accuracy_score(y_test, spline_probabilities >= 0.5),
        ],
    },
    index=["overall_hit_rate_baseline", "polynomial_interactions", "spline_logistic_regression"],
)

metrics

,log_loss,brier_score,accuracy_at_0.5
overall_hit_rate_baseline,0.624875,0.216642,0.682819
polynomial_interactions,0.528640,0.172199,0.751449
spline_logistic_regression,0.410585,0.131805,0.813819


In [31]:
# Step 8: Aggregate contact-level probabilities into a batter-level expected batting average.
# The spline model is the primary estimate because it performed best on the held-out week.
batter_summary = (
    test_data.groupby("batter")
    .agg(
        batted_balls=("is_hit", "size"),
        observed_batting_average=("is_hit", "mean"),
        expected_batting_average=("spline_predicted_hit_probability", "mean"),
        polynomial_expected_batting_average=("predicted_hit_probability", "mean"),
    )
    .sort_values(["batted_balls", "expected_batting_average"], ascending=False)
)

batter_summary.head(10)

,batted_balls,observed_batting_average,expected_batting_average,polynomial_expected_batting_average
batter,,,,
663538,24,0.375000,0.296629,0.323028
660271,23,0.565217,0.497739,0.459981
669394,23,0.391304,0.331497,0.318998
543760,23,0.347826,0.315154,0.338591
669364,22,0.590909,0.352139,0.352604
671277,22,0.363636,0.262143,0.297486
677594,21,0.380952,0.333311,0.384879
668939,21,0.238095,0.332873,0.318716
691406,21,0.285714,0.267162,0.277761
